# TOMATOES Price Analysis

This notebook loads TOMATOES price data from the CSV files in `data/`, combines the available days, and starts by calculating rolling volatility from the TOMATOES mid-price series.

Change `DATA_GLOB` if you want to narrow the input files.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

DATA_GLOB = "data/prices_round_*.csv"
PRODUCT = "TOMATOES"
ROLLING_WINDOW = 20


In [ ]:
price_paths = sorted(Path().glob(DATA_GLOB))
raw_prices = pd.concat((pd.read_csv(path, sep=";") for path in price_paths), ignore_index=True)

prices = (
    raw_prices.loc[raw_prices["product"] == PRODUCT]
    .copy()
    .sort_values(["day", "timestamp"])
    .reset_index(drop=True)
)

prices["event_index"] = range(len(prices))
price_paths, prices[["day", "timestamp", "product", "bid_price_1", "ask_price_1", "mid_price"]].head()


## Rolling Volatility

A common choice is to compute volatility on returns rather than on raw prices. Here we use percentage returns of the `mid_price`, then take a rolling standard deviation over the last `ROLLING_WINDOW` observations.


In [ ]:
prices["mid_return"] = prices["mid_price"].pct_change()
prices[f"rolling_volatility_{ROLLING_WINDOW}"] = (
    prices["mid_return"].rolling(ROLLING_WINDOW).std() * (ROLLING_WINDOW ** 0.5)
)

prices[[
    "day",
    "timestamp",
    "mid_price",
    "mid_return",
    f"rolling_volatility_{ROLLING_WINDOW}",
]].head(25)


In [ ]:
prices[f"rolling_volatility_{ROLLING_WINDOW}"].describe()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

axes[0].plot(prices["event_index"], prices["mid_price"], color="tab:blue", linewidth=1.5)
axes[0].set_title(f"{PRODUCT} Mid Price")
axes[0].set_ylabel("Mid Price")

axes[1].plot(
    prices["event_index"],
    prices[f"rolling_volatility_{ROLLING_WINDOW}"],
    color="tab:red",
    linewidth=1.5,
)
axes[1].set_title(f"{PRODUCT} Rolling Volatility ({ROLLING_WINDOW} observations)")
axes[1].set_xlabel("Observation Index")
axes[1].set_ylabel("Volatility")

plt.tight_layout()
plt.show()


## Trend Persistence From Log Returns

To test whether TOMATOES trends tick to tick, compute log returns within each day, take the sign of each return, and compare it to the previous tick's sign. If the same-sign percentage is materially above `50%`, that suggests short-horizon persistence. Below `50%` suggests the series is more choppy or mean-reverting than trending.


In [ ]:
prices["log_return"] = prices.groupby("day")["mid_price"].transform(lambda s: np.log(s).diff())
prices["log_return_sign"] = np.sign(prices["log_return"])
prices["prev_log_return_sign"] = prices.groupby("day")["log_return_sign"].shift(1)

same_sign_mask = (
    prices["log_return_sign"].notna()
    & prices["prev_log_return_sign"].notna()
    & prices["log_return_sign"].ne(0)
    & prices["prev_log_return_sign"].ne(0)
)

trend_summary = pd.DataFrame(
    {
        "same_sign_pct": [
            (prices.loc[same_sign_mask, "log_return_sign"] == prices.loc[same_sign_mask, "prev_log_return_sign"]).mean() * 100
        ],
        "opposite_sign_pct": [
            (prices.loc[same_sign_mask, "log_return_sign"] != prices.loc[same_sign_mask, "prev_log_return_sign"]).mean() * 100
        ],
        "zero_involved_pct": [
            ((prices["log_return_sign"].eq(0) | prices["prev_log_return_sign"].eq(0)).fillna(False)).mean() * 100
        ],
        "observations_used": [int(same_sign_mask.sum())],
    },
    index=["overall"],
).round(2)

trend_by_day = (
    prices.loc[same_sign_mask, ["day", "log_return_sign", "prev_log_return_sign"]]
    .assign(same_sign=lambda df: df["log_return_sign"] == df["prev_log_return_sign"])
    .groupby("day")["same_sign"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "same_sign_pct", "count": "observations_used"})
)
trend_by_day["same_sign_pct"] = (trend_by_day["same_sign_pct"] * 100).round(2)

print(
    "TOMATOES shows short-horizon trend persistence."
    if trend_summary.loc["overall", "same_sign_pct"] > 50
    else "TOMATOES does not show short-horizon trend persistence; sign flips are more common than same-direction moves."
)

trend_summary, trend_by_day
